# Evaluate sparse-structure mappers

Evaluate every mapper in `MAPPER_NAMES` on the same deterministic set of eight toys4k pairs per shape. Metrics are computed per pair and then averaged across pairs. This notebook only displays an in-memory DataFrame.

In [ ]:
from pathlib import Path

import pandas as pd
import torch
import torch.nn.functional as F
import trellis2.models as trellis2_models
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

from dataset import DirectFileLoadDataset, MultiScaleMixedPairSampler, build_catalog, pair_collate
from symtrellis.geometry import t_abs2grid
from symtrellis.mapper import from_pretrained


In [ ]:
MAPPER_NAMES = [
    "trellis2/sparse_structure/swin3d/legacy",
    "trellis2/sparse_structure/neighbor_graph/finetune",
]

TOYS4K_DATA_DIR = Path(
    "/mnt/scratch/trellis500k/toys4k/trellis2/multi_slats/"
    "ss_enc_conv3d_16l8_fp16_sslatentres_16_occres_64_s1_r16_p4"
)
DECODER_PRETRAINED_PATH = "microsoft/TRELLIS-image-large/ckpts/ss_dec_conv3d_16l8_fp16"

PAIRS_PER_SHAPE = 8
EVAL_BATCH_SIZE = 16
EVAL_SEED = 42
NUM_WORKERS = 8
PIN_MEMORY = True
PREFETCH_FACTOR = 2
PERSISTENT_WORKERS = True
DEVICE = torch.device("cuda")

GRID_SIZE = 16
NUM_SCALES = 1
NUM_ROTS = 16
NUM_PERTS = 4
SAME_ROT_DIFF_PERT_RATIO = 0.2
DST_INPUT_NORM_THRESHOLD = 1.5


## Fixed evaluation pairs

The index sampler uses a unit batch only to generate exactly `num_shapes * PAIRS_PER_SHAPE` deterministic indices. The DataLoader then groups those indices into evaluation batches and retains the final partial batch.

In [ ]:
shape_paths = build_catalog([TOYS4K_DATA_DIR], seed=EVAL_SEED)
eval_dataset = DirectFileLoadDataset(
    shape_paths=shape_paths,
    grid_size=GRID_SIZE,
    num_scale=NUM_SCALES,
    num_rots=NUM_ROTS,
    num_perts=NUM_PERTS,
)
index_sampler = MultiScaleMixedPairSampler(
    num_shapes=eval_dataset.num_shapes,
    num_scale=eval_dataset.num_scale,
    num_rots=eval_dataset.num_rots,
    num_perts=eval_dataset.num_perts,
    batch_size=1,
    num_batch_per_epoch=eval_dataset.num_shapes * PAIRS_PER_SHAPE,
    rank=0,
    world_size=1,
    seed=EVAL_SEED,
    same_rot_diff_pert_ratio=SAME_ROT_DIFF_PERT_RATIO,
)
pair_indices = [batch[0] for batch in index_sampler]
eval_loader = DataLoader(
    eval_dataset,
    batch_size=EVAL_BATCH_SIZE,
    sampler=pair_indices,
    drop_last=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    persistent_workers=PERSISTENT_WORKERS,
    prefetch_factor=PREFETCH_FACTOR,
    collate_fn=pair_collate,
)

print(f"shapes: {eval_dataset.num_shapes:,}")
print(f"pairs:  {len(pair_indices):,}")
print(f"batches: {len(eval_loader):,}")


In [ ]:
decoder = trellis2_models.from_pretrained(DECODER_PRETRAINED_PATH)
decoder.convert_to_fp32()
decoder.float().eval().to(DEVICE)
for parameter in decoder.parameters():
    parameter.requires_grad_(False)


In [ ]:
def compute_ss_feature_metrics(prediction, target):
    diff = prediction.float() - target.float()
    per_row_l1 = diff.abs().mean(dim=1)
    per_row_l2 = diff.square().mean(dim=1)
    per_row_cosine = F.cosine_similarity(prediction.float(), target.float(), dim=1)
    target_norm = target.float().norm(dim=1)

    filtered = target_norm >= DST_INPUT_NORM_THRESHOLD
    filtered_cosine = per_row_cosine[filtered].mean()
    unfiltered_cosine = per_row_cosine.mean()
    target_norm_sum = target_norm.sum().clamp_min(1e-12)
    norm_weighted_cosine = (per_row_cosine * target_norm).sum() / target_norm_sum

    return {
        "feature_filtered_l1": per_row_l1[filtered].mean(),
        "feature_filtered_l2": per_row_l2[filtered].mean(),
        "feature_filtered_cosine": filtered_cosine,
        "feature_filtered_cosine_distance": 1.0 - filtered_cosine,
        "feature_filter_keep_ratio": filtered.float().mean(),
        "feature_unfiltered_l1": per_row_l1.mean(),
        "feature_unfiltered_l2": per_row_l2.mean(),
        "feature_unfiltered_cosine": unfiltered_cosine,
        "feature_unfiltered_cosine_distance": 1.0 - unfiltered_cosine,
        "feature_norm_weighted_l1": (per_row_l1 * target_norm).sum() / target_norm_sum,
        "feature_norm_weighted_l2": (per_row_l2 * target_norm).sum() / target_norm_sum,
        "feature_norm_weighted_cosine": norm_weighted_cosine,
        "feature_norm_weighted_cosine_distance": 1.0 - norm_weighted_cosine,
    }


@torch.no_grad()
def evaluate_ss_mapper(model, loader, decoder):
    metric_sums = {}
    num_evaluated_samples = 0

    model.eval()
    for batch in tqdm(loader, desc="evaluate", leave=False):
        batch = {name: tensor.to(DEVICE, non_blocking=True) for name, tensor in batch.items()}
        t_grid = t_abs2grid(
            t_abs=batch["t_dst2src"],
            O=batch["O_dst2src"],
            grid_size=GRID_SIZE,
        )
        coeff = model(
            coords_src=batch["coords_src"],
            coords_dst=batch["coords_dst"],
            O_dst2src=batch["O_dst2src"],
            t_dst2src=t_grid,
            s_dst2src=batch["s_dst2src"],
        )
        prediction = coeff.apply(batch["feats_src"].to(dtype=coeff.dtype)).float()
        target = batch["feats_dst"].float()

        batch_size = batch["O_dst2src"].shape[0]
        latent_dim = target.shape[1]
        destination_batch_ids = batch["coords_dst"][:, 0].long()
        destination_xyz = batch["coords_dst"][:, 1:].long()
        dense_target = target.new_zeros((batch_size, latent_dim, GRID_SIZE, GRID_SIZE, GRID_SIZE))
        dense_prediction = prediction.new_zeros((batch_size, latent_dim, GRID_SIZE, GRID_SIZE, GRID_SIZE))
        dense_target[
            destination_batch_ids,
            :,
            destination_xyz[:, 0],
            destination_xyz[:, 1],
            destination_xyz[:, 2],
        ] = target
        dense_prediction[
            destination_batch_ids,
            :,
            destination_xyz[:, 0],
            destination_xyz[:, 1],
            destination_xyz[:, 2],
        ] = prediction

        target_logits = decoder(dense_target).float()
        prediction_logits = decoder(dense_prediction).float()
        decoder_l2 = (prediction_logits - target_logits).square().flatten(1).mean(dim=1)
        target_occupied = target_logits > 0
        prediction_occupied = prediction_logits > 0
        intersection = (target_occupied & prediction_occupied).flatten(1).sum(dim=1).float()
        union = (target_occupied | prediction_occupied).flatten(1).sum(dim=1).float()
        decoder_iou = intersection / union.clamp_min(1.0)

        for sample_id in range(batch_size):
            row_mask = destination_batch_ids == sample_id
            sample_metrics = compute_ss_feature_metrics(prediction[row_mask], target[row_mask])
            sample_metrics["decoder_l2"] = decoder_l2[sample_id]
            sample_metrics["decoder_iou"] = decoder_iou[sample_id]
            for name, value in sample_metrics.items():
                metric_sums[name] = metric_sums.get(name, value.new_zeros((), dtype=torch.float64)) + value.double()
            num_evaluated_samples += 1

    result = {"num_evaluated_samples": num_evaluated_samples}
    result.update({name: (value / num_evaluated_samples).item() for name, value in metric_sums.items()})
    return result


In [ ]:
rows = []
for model_name in tqdm(MAPPER_NAMES, desc="models"):
    print(f"Evaluating {model_name}")
    model = from_pretrained(model_name, device=DEVICE).eval()
    metrics = evaluate_ss_mapper(model, eval_loader, decoder)
    rows.append({"model_name": model_name, **metrics})
    del model
    torch.cuda.empty_cache()

evaluation_df = pd.DataFrame(rows)
evaluation_df
